In [8]:
import pandas as pd
from typing import Dict, List, Tuple, Set, Any, Optional
import json

In [10]:
def create_ablated_sentences(sentence_row: pd.Series, lexicon) -> List[Dict[str, Any]]:
    """Create ablated versions of a sentence by substituting one word at a time in SVO order.
    
    Args:
        sentence_row: A pandas Series containing 'input', 'lang', 'plural', 'subj', 'obj', 'verb'
        
    Returns:
        List of dictionaries containing ablated sentences with metadata
    """
        
    # Get source and target languages
    src_lang = sentence_row['lang']
    tgt_lang = 'nl' if src_lang == 'fr' else 'fr'
    
    # Parse the original sentence components
    plural = sentence_row['plural']
    det_type = "pl" if plural else "sgl"
    
    # Get determiners
    src_det = lexicon["DET"][src_lang][det_type]
    tgt_det = lexicon["DET"][tgt_lang][det_type]
    
    # Get nouns using parallel indices
    src_nouns = list(lexicon["NOUNS"][src_lang].keys())
    tgt_nouns = list(lexicon["NOUNS"][tgt_lang].keys())
    
    # Get subject translations
    subj_idx = src_nouns.index(sentence_row['subj'])
    tgt_subj_key = tgt_nouns[subj_idx]
    src_subj = lexicon["NOUNS"][src_lang][sentence_row['subj']][det_type]
    tgt_subj = lexicon["NOUNS"][tgt_lang][tgt_subj_key][det_type]
    
    # Get object translations
    obj_idx = src_nouns.index(sentence_row['obj'])
    tgt_obj_key = tgt_nouns[obj_idx]
    src_obj = lexicon["NOUNS"][src_lang][sentence_row['obj']][det_type]
    tgt_obj = lexicon["NOUNS"][tgt_lang][tgt_obj_key][det_type]
    
    # Get verb forms
    verb_key = sentence_row['verb']
    src_verbs = list(lexicon["VERBS"][src_lang].keys())
    tgt_verbs = list(lexicon["VERBS"][tgt_lang].keys())
    verb_idx = src_verbs.index(verb_key)
    tgt_verb_key = tgt_verbs[verb_idx]
    
    def get_verb_form(verb_info: dict, lang: str, is_plural: bool) -> str:
        if lang == "nl":
            return verb_info["present"]["zijpl" if is_plural else "hij"]
        else:  # fr
            return verb_info["present"]["ils" if is_plural else "il"]
    
    src_verb = get_verb_form(lexicon["VERBS"][src_lang][verb_key], src_lang, plural)
    tgt_verb = get_verb_form(lexicon["VERBS"][tgt_lang][tgt_verb_key], tgt_lang, plural)
    
    # Create ablated versions
    ablated = []
    
    # Original sentence
    base_sentence = f"{src_det} {src_subj} {src_verb} {src_det} {src_obj}"
    ablated.append({
        'input': base_sentence,
        'target': sentence_row['target'],
        'lang': src_lang,
        'plural': plural,
        'ablation': 'none',
        'subj': sentence_row['subj'],
        'obj': sentence_row['obj'],
        'verb': verb_key
    })
    
    # 1. Subject ablation (determinant + noun)
    subj_ablated = f"{tgt_det} {tgt_subj} {src_verb} {src_det} {src_obj}"
    ablated.append({
        'input': subj_ablated,
        'target': sentence_row['target'],
        'lang': src_lang,
        'plural': plural,
        'ablation': 'subject',
        'subj': sentence_row['subj'],
        'obj': sentence_row['obj'],
        'verb': verb_key
    })
    
    # 2. Verb ablation
    verb_ablated = f"{src_det} {src_subj} {tgt_verb} {src_det} {src_obj}"
    ablated.append({
        'input': verb_ablated,
        'target': sentence_row['target'],
        'lang': src_lang,
        'plural': plural,
        'ablation': 'verb',
        'subj': sentence_row['subj'],
        'obj': sentence_row['obj'],
        'verb': verb_key
    })
    
    # 3. Object ablation (determinant + noun)
    obj_ablated = f"{src_det} {src_subj} {src_verb} {tgt_det} {tgt_obj}"
    ablated.append({
        'input': obj_ablated,
        'target': sentence_row['target'],
        'lang': src_lang,
        'plural': plural,
        'ablation': 'object',
        'subj': sentence_row['subj'],
        'obj': sentence_row['obj'],
        'verb': verb_key
    })
    
    return ablated

In [11]:
"""Create an ablated version of the dataset with one-word substitutions.

Args:
    input_df: Input DataFrame containing sentences to ablate
    
Returns:
    DataFrame containing original and ablated sentences
"""
input_df = pd.read_csv("/n/home06/drooryck/codeswitching-llms/july_aug_exp/scripts/data/new_test.csv")
with open("/n/home06/drooryck/codeswitching-llms/july_aug_exp/scripts/data/lexicon_new.json", 'r') as f:
    lexicon = json.load(f)
    
all_ablated = []
for _, row in input_df.iterrows():
    ablated_sentences = create_ablated_sentences(row, lexicon)
    all_ablated.extend(ablated_sentences)

abl = pd.DataFrame(all_ablated)
abl.head(20)

,input,target,lang,plural,ablation,subj,obj,verb
0,le chien voit le loup,le chien a vu le loup,fr,False,none,chien,loup,voit
1,de hond voit le loup,le chien a vu le loup,fr,False,subject,chien,loup,voit
2,le chien ziet le loup,le chien a vu le loup,fr,False,verb,chien,loup,voit
3,le chien voit de leeuw,le chien a vu le loup,fr,False,object,chien,loup,voit
4,les chiens voient les loups,les chiens a vu les loups,fr,True,none,chien,loup,voit
5,de honden voient les loups,les chiens a vu les loups,fr,True,subject,chien,loup,voit
6,les chiens zien les loups,les chiens a vu les loups,fr,True,verb,chien,loup,voit
7,les chiens voient de leeuwen,les chiens a vu les loups,fr,True,object,chien,loup,voit
8,le chien aime le loup,le chien a aimé le loup,fr,False,none,chien,loup,aime
9,de hond aime le loup,le chien a aimé le loup,fr,False,subject,chien,loup,aime


In [16]:
first_row = input_df.iloc[0]

src_lang = first_row['lang']
tgt_lang = 'nl' if src_lang == 'fr' else 'fr'

plural = first_row['plural']

verb_key = first_row['verb']
src_verbs = list(lexicon["VERBS"][src_lang].keys())
tgt_verbs = list(lexicon["VERBS"][tgt_lang].keys())
verb_idx = src_verbs.index(verb_key)
tgt_verb_key = tgt_verbs[verb_idx]

def get_verb_form(verb_info: dict, lang: str, is_plural: bool) -> str:
    if lang == "nl":
        return verb_info["present"]["zijpl" if is_plural else "hij"]
    else:  # fr
        return verb_info["present"]["ils" if is_plural else "il"]

src_verb = get_verb_form(lexicon["VERBS"][src_lang][verb_key], src_lang, plural)
tgt_verb = get_verb_form(lexicon["VERBS"][tgt_lang][tgt_verb_key], tgt_lang, plural)

print(src_verb)
print(tgt_verb)

voit
ziet


In [17]:
sentence_row = input_df.iloc[0]

# Get source and target languages
src_lang = sentence_row['lang']
tgt_lang = 'nl' if src_lang == 'fr' else 'fr'

# Parse the original sentence components
plural = sentence_row['plural']
det_type = "pl" if plural else "sgl"

# Get determiners
src_det = lexicon["DET"][src_lang][det_type]
tgt_det = lexicon["DET"][tgt_lang][det_type]

# Get nouns using parallel indices
src_nouns = list(lexicon["NOUNS"][src_lang].keys())
tgt_nouns = list(lexicon["NOUNS"][tgt_lang].keys())

# Get subject translations
subj_idx = src_nouns.index(sentence_row['subj'])
tgt_subj_key = tgt_nouns[subj_idx]
src_subj = lexicon["NOUNS"][src_lang][sentence_row['subj']][det_type]
tgt_subj = lexicon["NOUNS"][tgt_lang][tgt_subj_key][det_type]

# Get object translations
obj_idx = src_nouns.index(sentence_row['obj'])
tgt_obj_key = tgt_nouns[obj_idx]
src_obj = lexicon["NOUNS"][src_lang][sentence_row['obj']][det_type]
tgt_obj = lexicon["NOUNS"][tgt_lang][tgt_obj_key][det_type]

# Get verb forms
verb_key = sentence_row['verb']
src_verbs = list(lexicon["VERBS"][src_lang].keys())
tgt_verbs = list(lexicon["VERBS"][tgt_lang].keys())
verb_idx = src_verbs.index(verb_key)
tgt_verb_key = tgt_verbs[verb_idx]

def get_verb_form(verb_info: dict, lang: str, is_plural: bool) -> str:
    if lang == "nl":
        return verb_info["present"]["zijpl" if is_plural else "hij"]
    else:  # fr
        return verb_info["present"]["ils" if is_plural else "il"]

src_verb = get_verb_form(lexicon["VERBS"][src_lang][verb_key], src_lang, plural)
tgt_verb = get_verb_form(lexicon["VERBS"][tgt_lang][tgt_verb_key], tgt_lang, plural)

# Create ablated versions
ablated = []

# Original sentence
base_sentence = f"{src_det} {src_subj} {src_verb} {src_det} {src_obj}"
ablated.append({
    'input': base_sentence,
    'target': sentence_row['target'],
    'lang': src_lang,
    'plural': plural,
    'ablation': 'none',
    'subj': sentence_row['subj'],
    'obj': sentence_row['obj'],
    'verb': verb_key
})

# 1. Subject ablation (determinant + noun)
subj_ablated = f"{tgt_det} {tgt_subj} {src_verb} {src_det} {src_obj}"
ablated.append({
    'input': subj_ablated,
    'target': sentence_row['target'],
    'lang': src_lang,
    'plural': plural,
    'ablation': 'subject',
    'subj': sentence_row['subj'],
    'obj': sentence_row['obj'],
    'verb': verb_key
})

# 2. Verb ablation
verb_ablated = f"{src_det} {src_subj} {tgt_verb} {src_det} {src_obj}"
ablated.append({
    'input': verb_ablated,
    'target': sentence_row['target'],
    'lang': src_lang,
    'plural': plural,
    'ablation': 'verb',
    'subj': sentence_row['subj'],
    'obj': sentence_row['obj'],
    'verb': verb_key
})

# 3. Object ablation (determinant + noun)
obj_ablated = f"{src_det} {src_subj} {src_verb} {tgt_det} {tgt_obj}"
ablated.append({
    'input': obj_ablated,
    'target': sentence_row['target'],
    'lang': src_lang,
    'plural': plural,
    'ablation': 'object',
    'subj': sentence_row['subj'],
    'obj': sentence_row['obj'],
    'verb': verb_key
})

ablated

[{'input': 'le chien voit le loup',
  'target': 'le chien a vu le loup',
  'lang': 'fr',
  'plural': np.False_,
  'ablation': 'none',
  'subj': 'chien',
  'obj': 'loup',
  'verb': 'voit'},
 {'input': 'de hond voit le loup',
  'target': 'le chien a vu le loup',
  'lang': 'fr',
  'plural': np.False_,
  'ablation': 'subject',
  'subj': 'chien',
  'obj': 'loup',
  'verb': 'voit'},
 {'input': 'le chien ziet le loup',
  'target': 'le chien a vu le loup',
  'lang': 'fr',
  'plural': np.False_,
  'ablation': 'verb',
  'subj': 'chien',
  'obj': 'loup',
  'verb': 'voit'},
 {'input': 'le chien voit de leeuw',
  'target': 'le chien a vu le loup',
  'lang': 'fr',
  'plural': np.False_,
  'ablation': 'object',
  'subj': 'chien',
  'obj': 'loup',
  'verb': 'voit'}]

In [23]:
import torch
from transformers import GPT2LMHeadModel, AutoTokenizer
from transformers import logging as transformers_logging
from typing import Dict, List, Any

transformers_logging.set_verbosity_error()

def run_inference_on_ablated(model_path: str, test_df: pd.DataFrame) -> List[Dict[str, Any]]:
    """Run inference on ablated test data.
    
    Args:
        model_path: Path to the model directory (containing model and tokenizer)
        test_df: DataFrame containing ablated test sentences
        
    Returns:
        List of prediction dictionaries
    """
    # Load model and tokenizer
    #print('load model and tokenizer')
    model = GPT2LMHeadModel.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    # Set device
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    
    predictions = []
    
    for row in test_df.itertuples():
        prompt = f"<sos> {row.input} <sep>"
        #print(f"nu aan: {prompt}")
        
        with torch.no_grad():
            inputs = tokenizer(prompt, return_tensors="pt").to(device)
            outputs = model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False,
                num_beams=4,
                eos_token_id=tokenizer.eos_token_id
            )
            
        pred = tokenizer.decode(outputs[0], skip_special_tokens=False)
        pred = pred.split("<sep>")[1].replace("<eos>", "").strip()
        
        predictions.append({
            'language': row.lang,
            'input': row.input,
            'gold': row.target,
            'prediction': pred,
            'ablation': row.ablation  # Include ablation type in results
        })
    
    return predictions

# Use it on your ablated data
model_path = "/n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep12.3/p50_run1/final"
predictions = run_inference_on_ablated(model_path, abl)

# Convert predictions to DataFrame and save
pred_df = pd.DataFrame(predictions)
pred_df.to_csv("/n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep12.3/p50_run1/ablation_predictions.csv", index=False)

# Show some examples
print("\nExample predictions:")
for i in range(4):  # Show first sentence with all its ablations
    row = pred_df.iloc[i]
    print(f"\nAblation: {row['ablation']}")
    print(f"Input: {row['input']}")
    print(f"Gold: {row['gold']}")
    print(f"Prediction: {row['prediction']}")


Example predictions:

Ablation: none
Input: le chien voit le loup
Gold: le chien a vu le loup
Prediction: le chien a vu le loup

Ablation: subject
Input: de hond voit le loup
Gold: le chien a vu le loup
Prediction: le hond heeft de loup loup

Ablation: verb
Input: le chien ziet le loup
Gold: le chien a vu le loup
Prediction: le chien a porté le loup

Ablation: object
Input: le chien voit de leeuw
Gold: le chien a vu le loup
Prediction: de chien heeft de leeuw getroost


In [24]:
abl.to_csv("/n/home06/drooryck/codeswitching-llms/july_aug_exp/scripts/data/ablated_test.csv", index=False)

In [25]:
from pathlib import Path
import os

base_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9")
run_dirs = list(base_dir.glob("p*_run*"))

print(f"Found {len(run_dirs)} run directories")
print("\nChecking contents:")
for run_dir in sorted(run_dirs):
    final_dir = run_dir / "final"
    if not final_dir.exists():
        print(f"\n{run_dir.name}: No final directory")
        continue
        
    files = list(final_dir.glob("*"))
    if not files:
        print(f"\n{run_dir.name}: Final directory is empty")
        continue
        
    required_files = ["config.json", "model.safetensors", "tokenizer.json"]
    missing = [f for f in required_files if not (final_dir / f).exists()]
    
    if missing:
        print(f"\n{run_dir.name}: Missing files: {missing}")
        print(f"Has files: {[f.name for f in files]}")
    else:
        print(f"\n{run_dir.name}: ✓ All required files present")
        model_size = (final_dir / "model.safetensors").stat().st_size / (1024*1024)  # Convert to MB
        print(f"Model size: {model_size:.1f}MB")

Found 41 run directories

Checking contents:

p000_run3: ✓ All required files present
Model size: 50.5MB

p001_run3: ✓ All required files present
Model size: 50.5MB

p005_run3: ✓ All required files present
Model size: 50.5MB

p010_run3: ✓ All required files present
Model size: 50.5MB

p015_run3: ✓ All required files present
Model size: 50.5MB

p020_run3: ✓ All required files present
Model size: 50.5MB

p025_run3: ✓ All required files present
Model size: 50.5MB

p030_run3: ✓ All required files present
Model size: 50.5MB

p040_run3: ✓ All required files present
Model size: 50.5MB

p050_run3: ✓ All required files present
Model size: 50.5MB

p075_run3: ✓ All required files present
Model size: 50.5MB

p1000_run3: ✓ All required files present
Model size: 50.5MB

p100_run3: ✓ All required files present
Model size: 50.5MB

p150_run3: ✓ All required files present
Model size: 50.5MB

p200_run3: ✓ All required files present
Model size: 50.5MB

p250_run3: ✓ All required files present
Model size: 5

In [27]:
df = pd.read_csv("/n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/p990_run3/ablation_predictions.csv")

subset = df[df['ablation'] == 'subject'].head(20)
subset

,language,input,gold,prediction,ablation
1,fr,de hond voit le loup,le chien a vu le loup,le hond heeft de loup vu,subject
5,fr,de honden voient les loups,les chiens a vu les loups,les honden a vu les loups,subject
9,fr,de hond aime le loup,le chien a aimé le loup,le hond heeft de loup aimé,subject
13,fr,de honden aiment les loups,les chiens a aimé les loups,les honden a aimé les loups,subject
17,fr,de hond soigne le loup,le chien a soigné le loup,le hond heeft de loup soigné,subject
21,fr,de honden soignent les loups,les chiens a soigné les loups,les honden a soigné les loups,subject
25,fr,de hond poursuit le loup,le chien a poursuivi le loup,le hond a poursuivi le loup,subject
29,fr,de honden poursuivent les loups,les chiens a poursuivi les loups,les honden a poursuivi les loups,subject
33,fr,de hond réconforte le loup,le chien a réconforté le loup,le hond heeft de loup réconforté,subject
37,fr,de honden réconfortent les loups,les chiens a réconforté les loups,les honden a réconforté les loups,subject


In [28]:
subset = df[df['ablation'] == 'object'].head(20)
subset

,language,input,gold,prediction,ablation
3,fr,le chien voit de leeuw,le chien a vu le loup,le chien a vu le leeuw,object
7,fr,les chiens voient de leeuwen,les chiens a vu les loups,les chiens a vu les leeuwen,object
11,fr,le chien aime de leeuw,le chien a aimé le loup,le chien a aimé le leeuw,object
15,fr,les chiens aiment de leeuwen,les chiens a aimé les loups,les chiens a aimé les leeuwen,object
19,fr,le chien soigne de leeuw,le chien a soigné le loup,le chien a soigné le leeuw,object
23,fr,les chiens soignent de leeuwen,les chiens a soigné les loups,les chiens a soigné les soigné,object
27,fr,le chien poursuit de leeuw,le chien a poursuivi le loup,le chien a poursuivi le leeuw,object
31,fr,les chiens poursuivent de leeuwen,les chiens a poursuivi les loups,les chiens a poursuivi les leeuwen,object
35,fr,le chien réconforte de leeuw,le chien a réconforté le loup,le chien a réconforté le leeuw,object
39,fr,les chiens réconfortent de leeuwen,les chiens a réconforté les loups,les chiens a réconforté les leeuwen,object


In [29]:
subset = df[df['ablation'] == 'verb'].head(20)
subset

,language,input,gold,prediction,ablation
2,fr,le chien ziet le loup,le chien a vu le loup,le chien a gezien le loup,verb
6,fr,les chiens zien les loups,les chiens a vu les loups,les chiens a gezien les loups,verb
10,fr,le chien mag le loup,le chien a aimé le loup,le chien a gemogen le loup,verb
14,fr,les chiens mogen les loups,les chiens a aimé les loups,les chiens a gemogen les loups,verb
18,fr,le chien verzorgt le loup,le chien a soigné le loup,le chien a verzorgd le loup,verb
22,fr,les chiens verzorgen les loups,les chiens a soigné les loups,les chiens a verzorgd les loups,verb
26,fr,le chien achtervolgt le loup,le chien a poursuivi le loup,le chien a achtervolgd le loup,verb
30,fr,les chiens achtervolgen les loups,les chiens a poursuivi les loups,les chiens a achtervolgd les loups,verb
34,fr,le chien troost le loup,le chien a réconforté le loup,le chien a getroost le loup,verb
38,fr,les chiens troosten les loups,les chiens a réconforté les loups,les chiens a getroost les loups,verb


In [2]:
import pandas as pd
df = pd.read_csv("/n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/p990_run3/ablation_predictions.csv")

subset = df[df['ablation'] == 'none'].head(20)
subset

,language,input,gold,prediction,ablation
0,fr,le chien voit le loup,le chien a vu le loup,le chien a vu le loup,none
4,fr,les chiens voient les loups,les chiens a vu les loups,les chiens a vu les loups,none
8,fr,le chien aime le loup,le chien a aimé le loup,le chien a aimé le loup,none
12,fr,les chiens aiment les loups,les chiens a aimé les loups,les chiens a aimé les loups,none
16,fr,le chien soigne le loup,le chien a soigné le loup,le chien a soigné le loup,none
20,fr,les chiens soignent les loups,les chiens a soigné les loups,les chiens a soigné les loups,none
24,fr,le chien poursuit le loup,le chien a poursuivi le loup,le chien a poursuivi le loup,none
28,fr,les chiens poursuivent les loups,les chiens a poursuivi les loups,les chiens a poursuivi les loups,none
32,fr,le chien réconforte le loup,le chien a réconforté le loup,le chien a réconforté le loup,none
36,fr,les chiens réconfortent les loups,les chiens a réconforté les loups,les chiens a réconforté les loups,none


In [44]:
df = pd.read_csv("/n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/p010_run3/ablation_predictions.csv")

subset = df[df['ablation'] == 'object'].tail(20)
subset

,language,input,gold,prediction,ablation
20659,nl,de vijand negeert le professeur,de vijand heeft de leraar genegeerd,de vijand heeft de professeur genegeerd,object
20663,nl,de vijanden negeren les professeurs,de vijanden heeft de leraars genegeerd,de vijanden heeft de professeurs genegeerd,object
20667,nl,de vijand helpt le ami,de vijand heeft de vriend geholpen,de vijand heeft de ami geholpen,object
20671,nl,de vijanden helpen les amis,de vijanden heeft de vrienden geholpen,de vijanden heeft de amis geholpen,object
20675,nl,de vijand volgt le ami,de vijand heeft de vriend gevolgd,de vijand heeft de ami gevolgd,object
20679,nl,de vijanden volgen les amis,de vijanden heeft de vrienden gevolgd,de vijanden heeft de amis gevolgd,object
20683,nl,de vijand slaat le ami,de vijand heeft de vriend geslagen,de vijand heeft de ami geslagen,object
20687,nl,de vijanden slaan les amis,de vijanden heeft de vrienden geslagen,de vijanden heeft de amis geslagen,object
20691,nl,de vijand draagt le ami,de vijand heeft de vriend gedragen,de vijand heeft de ami gedragen,object
20695,nl,de vijanden dragen les amis,de vijanden heeft de vrienden gedragen,de vijanden heeft de amis gedragen,object


## MOSTLY DUTCH

In [3]:
import pandas as pd
df_2 = pd.read_csv("/n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/p010_run3/ablation_predictions.csv")

subset = df_2[df_2['ablation'] == 'subject'].head(50)
subset

,language,input,gold,prediction,ablation
1,fr,de hond voit le loup,le chien a vu le loup,de hond heeft de loup vu,subject
5,fr,de honden voient les loups,les chiens a vu les loups,de honden heeft de loups vu,subject
9,fr,de hond aime le loup,le chien a aimé le loup,de hond heeft de loup aimé,subject
13,fr,de honden aiment les loups,les chiens a aimé les loups,de honden heeft de loups aimé,subject
17,fr,de hond soigne le loup,le chien a soigné le loup,de hond heeft de loup soigné,subject
21,fr,de honden soignent les loups,les chiens a soigné les loups,de honden heeft de loups soigné,subject
25,fr,de hond poursuit le loup,le chien a poursuivi le loup,de hond heeft de loup poursuivi,subject
29,fr,de honden poursuivent les loups,les chiens a poursuivi les loups,de honden heeft de loups poursuivi,subject
33,fr,de hond réconforte le loup,le chien a réconforté le loup,de hond heeft de loup réconforté,subject
37,fr,de honden réconfortent les loups,les chiens a réconforté les loups,de honden heeft de loups réconforté,subject


In [12]:
from pathlib import Path
import pandas as pd
import json
import sys
project_root = Path("/n/home06/drooryck/codeswitching-llms")
sys.path.append(str(project_root))
from july_aug_exp.src.metrics import Metrics

# Initialize metrics with lexicon
metrics = Metrics(Path("/n/home06/drooryck/codeswitching-llms/july_aug_exp/scripts/data/lexicon_new.json"))

# Get all run directories
output_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/")
run_dirs = list(output_dir.glob("p*_run*"))

# Process each run directory
for run_dir in run_dirs:
    ablation_file = run_dir / "ablation_predictions.csv"
    if not ablation_file.exists():
        print(f"Skipping {run_dir}: no ablation predictions")
        continue
        
    preds_df = pd.read_csv(ablation_file)
    
    for ablation_type in ['none', 'subject', 'verb', 'object']:
        type_preds = preds_df[preds_df['ablation'] == ablation_type]
        
        predictions = [
            {
                'language': row['language'],
                'prediction': row['prediction'],
                'gold': row['gold'],
                'input': row['input']
            }
            for _, row in type_preds.iterrows()
        ]
        
        computed_metrics = metrics.compute_all_metrics(predictions)
        
        metrics_file = run_dir / f"ablation_{ablation_type}_metrics.json"
        with open(metrics_file, 'w') as f:
            json.dump(computed_metrics, f, indent=2)
    
    print(f"Saved metrics for {run_dir.name}")

print("Done!")

Saved metrics for p020_run3
Saved metrics for p350_run3
Saved metrics for p995_run3
Saved metrics for p1000_run3
Saved metrics for p970_run3
Saved metrics for p001_run3
Saved metrics for p100_run3
Saved metrics for p800_run3
Saved metrics for p960_run3
Saved metrics for p500_run3
Saved metrics for p450_run3
Saved metrics for p150_run3
Saved metrics for p950_run3
Saved metrics for p010_run3
Saved metrics for p975_run3
Saved metrics for p900_run3
Saved metrics for p050_run3
Saved metrics for p030_run3
Saved metrics for p300_run3
Saved metrics for p600_run3
Saved metrics for p550_run3
Saved metrics for p700_run3
Saved metrics for p999_run3
Saved metrics for p850_run3
Saved metrics for p000_run3
Saved metrics for p985_run3
Saved metrics for p025_run3
Saved metrics for p650_run3
Saved metrics for p015_run3
Saved metrics for p040_run3
Saved metrics for p980_run3
Saved metrics for p075_run3
Saved metrics for p400_run3
Saved metrics for p750_run3
Saved metrics for p250_run3
Saved metrics for p

In [2]:
from pathlib import Path
import pandas as pd
import json
from typing import List
from july_aug_exp.src.plotting import BilingualPlotter

def collect_ablation_results(run_dirs: List[Path]) -> pd.DataFrame:
    """Collect ablation metrics from multiple experiment runs.

    Args:
        run_dirs: List of experiment output directories

    Returns:
        DataFrame with aggregated results including ablation types
    """
    results = []

    for run_dir in run_dirs:
        # Extract run parameters from directory name
        dir_name = run_dir.name
        if dir_name.startswith("p") and "_run" in dir_name:
            parts = dir_name.split("_run")
            prop = int(parts[0][1:]) / 100.0
            run_id = int(parts[1])
        else:
            continue

        # Collect metrics for each ablation type
        for ablation_type in ['none', 'subject', 'verb', 'object']:
            metrics_file = run_dir / f"ablation_{ablation_type}_metrics.json"
            if not metrics_file.exists():
                print(f"No {ablation_type} metrics found in {run_dir}")
                continue

            with open(metrics_file, 'r') as f:
                metrics = json.load(f)

            result = {
                'prop': prop,
                'run_id': run_id,
                'run_dir': str(run_dir),
                'ablation': ablation_type,
                **metrics
            }
            results.append(result)

    return pd.DataFrame(results)

# Set up directories
output_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/")
run_dirs = list(output_dir.glob("p*_run*"))

# Collect all results
results_df = collect_ablation_results(run_dirs)

# Create plots for each ablation type
for ablation_type in ['none', 'subject', 'verb', 'object']:
    # Create plots directory for this ablation type
    plots_dir = output_dir / f"plots_{ablation_type}"
    plots_dir.mkdir(exist_ok=True)
    
    type_results = results_df[results_df['ablation'] == ablation_type]
    
    if not type_results.empty:
        print(f"\nCreating plots for {ablation_type} ablation")
        print(f"Found {len(type_results)} runs")
        
        # Create plotter with specific output directory for this ablation type
        plotter = BilingualPlotter(type_results, plots_dir)
        plotter.create_all_plots()
        print(f"Saved plots to {plots_dir}")
    else:
        print(f"No results found for {ablation_type} ablation")

print("\nDone!")


Creating plots for none ablation
Found 39 runs
Saved plots to /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/plots_none

Creating plots for subject ablation
Found 39 runs
Saved plots to /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/plots_subject

Creating plots for verb ablation
Found 39 runs
Saved plots to /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/plots_verb

Creating plots for object ablation
Found 39 runs
Saved plots to /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/plots_object

Done!


## you need to add option 2 with a separate ablation method

## you need to fix compute_all_metrics and figure out how to group metrics.
## pick up from cursor chat


## fix compute_all_metrics calls and where to pass around the ablation type.
## figure out how collect_ablation_results fits into my class structure
## 

## the plots should all have both french and dutch test sentence metrics on them so you can compare them in the same plot
## in general plotting needs a big cleanup

## sep 19

In [1]:
from pathlib import Path
import pandas as pd
import json
import sys
project_root = Path("/n/home06/drooryck/codeswitching-llms")
sys.path.append(str(project_root))
from july_aug_exp.src.metrics import Metrics

# Initialize metrics with lexicon
metrics = Metrics(Path("/n/home06/drooryck/codeswitching-llms/july_aug_exp/scripts/data/lexicon_new.json"))

# Get all run directories
output_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/")
run_dirs = list(output_dir.glob("p*_run*"))

# Process each run directory
for run_dir in run_dirs:
    ablation_file = run_dir / "ablation_predictions.csv"
    if not ablation_file.exists():
        print(f"Skipping {run_dir}: no ablation predictions")
        continue
        
    print(f"\nProcessing {run_dir.name}")
    preds_df = pd.read_csv(ablation_file)
    
    for ablation_type in ['none', 'subject', 'verb', 'object']:
        print(f"  Computing {ablation_type} metrics...")
        type_preds = preds_df[preds_df['ablation'] == ablation_type]
        
        predictions = [
            {
                'language': row['language'],
                'prediction': row['prediction'],
                'gold': row['gold'],
                'input': row['input'],
                'ablation': ablation_type
            }
            for _, row in type_preds.iterrows()
        ]
        
        # Compute metrics with ablation type
        computed_metrics = metrics.compute_all_metrics(predictions, ablation_type)
        
        # Save metrics with backup
        metrics_file = run_dir / f"ablation_{ablation_type}_metrics.json"
        with open(metrics_file, 'w') as f:
            json.dump(computed_metrics, f, indent=2)
            
        
        print(f"    Saved metrics to {metrics_file.name}")

print("\nDone!")


Processing p020_run3
  Computing none metrics...
    Saved metrics to ablation_none_metrics.json
  Computing subject metrics...
    Saved metrics to ablation_subject_metrics.json
  Computing verb metrics...
    Saved metrics to ablation_verb_metrics.json
  Computing object metrics...
    Saved metrics to ablation_object_metrics.json

Processing p350_run3
  Computing none metrics...
    Saved metrics to ablation_none_metrics.json
  Computing subject metrics...
    Saved metrics to ablation_subject_metrics.json
  Computing verb metrics...
    Saved metrics to ablation_verb_metrics.json
  Computing object metrics...
    Saved metrics to ablation_object_metrics.json

Processing p995_run3
  Computing none metrics...
    Saved metrics to ablation_none_metrics.json
  Computing subject metrics...
    Saved metrics to ablation_subject_metrics.json
  Computing verb metrics...
    Saved metrics to ablation_verb_metrics.json
  Computing object metrics...
    Saved metrics to ablation_object_metri

In [ ]:
## plotting

# Create plotter from your metrics
output_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/")
plotter = BilingualPlotter.create_plotter_from_run_metrics_dir(output_dir)

# Create all plots including ablation plots
plotter.create_all_plots()

# Or create specific ablation plots
plotter.plot_ablation_structure_metrics()
plotter.plot_ablation_word_tracking()
plotter.plot_aux_verb_consistency()

In [1]:
# First load the predictions
from pathlib import Path
import pandas as pd
import json
import sys
project_root = Path("/n/home06/drooryck/codeswitching-llms")
sys.path.append(str(project_root))
from july_aug_exp.src.metrics import Metrics

# Initialize metrics with lexicon
metrics = Metrics(Path("/n/home06/drooryck/codeswitching-llms/july_aug_exp/scripts/data/lexicon_new.json"))

# Load predictions
preds_df = pd.read_csv("/n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/p050_run3/ablation_predictions.csv")

# Filter for French sentences with no ablation
fr_none_preds = preds_df[(preds_df['language'] == 'fr') & (preds_df['ablation'] == 'none')]

# Apply determiner_metrics to each prediction and collect results
results = []
for _, row in fr_none_preds.iterrows():
    metrics_dict = metrics.determiner_metrics(row['prediction'], 'fr')
    results.append({
        'prediction': row['prediction'],
        'det_lang_correct': metrics_dict['det_lang_correct'],
        'det_agreement': metrics_dict['det_agreement']
    })

# Convert to DataFrame for analysis
results_df = pd.DataFrame(results)

# Print summary statistics
print("\nSummary of determiner metrics on French non-ablated sentences:")
print(f"Total sentences analyzed: {len(results_df)}")
print(f"Determiner language correct: {results_df['det_lang_correct'].mean():.2%}")
print(f"Determiner agreement correct: {results_df['det_agreement'].mean():.2%}")

# Print some example failures
print("\nExamples where determiner agreement failed:")
failures = results_df[results_df['det_agreement'] == False]
for _, row in failures.head(5).iterrows():
    print(f"\nPrediction: {row['prediction']}")


Summary of determiner metrics on French non-ablated sentences:
Total sentences analyzed: 2628
Determiner language correct: 100.00%
Determiner agreement correct: 100.00%

Examples where determiner agreement failed:


In [3]:
from pathlib import Path
import json
import pandas as pd
from july_aug_exp.src.dataset_manager import DatasetManager
from july_aug_exp.src.model_config import ModelConfig

# Load config directly from json
config_path = "/n/home06/drooryck/codeswitching-llms/july_aug_exp/configs/default_model.json"
with open(config_path) as f:
    config_dict = json.load(f)
config = ModelConfig(**config_dict)

# Initialize DatasetManager
data_manager = DatasetManager("/n/home06/drooryck/codeswitching-llms/july_aug_exp/scripts/data", config)

# Build tokenizer
tokenizer = data_manager.build_tokenizer()

# Get vocab size and show all tokens
vocab_size = len(tokenizer.get_vocab())
vocab = tokenizer.get_vocab()

print(f"Vocabulary size: {vocab_size}")
print("\nSpecial tokens:")
for token in ['<pad>', '<sos>', '<eos>', '<unk>', '<sep>']:
    if token in vocab:
        print(f"  {token}: {vocab[token]}")

print("\nSample of vocabulary tokens:")
sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])  # Sort by token ID
for token, id in sorted_vocab[:10]:
    print(f"  {token}: {id}")

# Test tokenization on a sample sentence
sample = "le chien a mangé le loup"
tokens = tokenizer(sample)
print(f"\nSample tokenization of '{sample}':")
print(f"Token IDs: {tokens['input_ids']}")
print(f"Decoded back: {tokenizer.decode(tokens['input_ids'])}")

# Count tokens by first letter to see distribution
from collections import Counter
first_letters = Counter(token[0] for token in vocab.keys() if len(token) > 0)
print("\nToken counts by first letter:")
for letter, count in sorted(first_letters.items()):
    print(f"  {letter}: {count}")

Vocabulary size: 199

Special tokens:
  <pad>: 0
  <sos>: 1
  <eos>: 2
  <unk>: 3
  <sep>: 198

Sample of vocabulary tokens:
  <pad>: 0
  <sos>: 1
  <eos>: 2
  <unk>: 3
  a: 4
  aangevallen: 5
  aap: 6
  achtervolgd: 7
  achtervolgen: 8
  achtervolgt: 9

Sample tokenization of 'le chien a mangé le loup':
Token IDs: [97, 35, 4, 109, 97, 103]
Decoded back: le chien a mangé le loup

Token counts by first letter:
  <: 5
  a: 18
  b: 3
  c: 12
  d: 9
  e: 9
  f: 7
  g: 17
  h: 7
  i: 3
  j: 4
  k: 4
  l: 8
  m: 11
  n: 2
  o: 9
  p: 10
  r: 13
  s: 14
  t: 6
  v: 20
  z: 6
  é: 2


In [4]:
from pathlib import Path
import pandas as pd
from july_aug_exp.src.dataset_manager import DatasetManager
from july_aug_exp.src.model_config import ModelConfig

# Load config
config_path = "/n/home06/drooryck/codeswitching-llms/july_aug_exp/configs/default_model.json"
with open(config_path) as f:
    config_dict = json.load(f)
config = ModelConfig(**config_dict)

# Initialize DatasetManager and build tokenizer
data_manager = DatasetManager("/n/home06/drooryck/codeswitching-llms/july_aug_exp/scripts/data", config)
tokenizer = data_manager.build_tokenizer()

# Load training data
train_df = pd.read_csv("/n/home06/drooryck/codeswitching-llms/july_aug_exp/scripts/data/new_train.csv")

# Count tokens in each example (input + target)
total_tokens = 0
for _, row in train_df.iterrows():
    # Format matches your training format: <sos> input <sep> target <eos>
    full_text = f"<sos> {row['input']} <sep> {row['target']} <eos>"
    tokens = tokenizer(full_text)['input_ids']
    total_tokens += len(tokens)

print(f"Total examples: {len(train_df)}")
print(f"Total tokens in training: {total_tokens}")
print(f"Average tokens per example: {total_tokens/len(train_df):.1f}")

# Chinchilla analysis
params = 13_218_304  # from our previous calculation
chinchilla_recommended_tokens = params * 20  # Chinchilla recommends 20x tokens to params

print(f"\nChinchilla analysis:")
print(f"Our parameters: {params:,}")
print(f"Our training tokens: {total_tokens:,}")
print(f"Chinchilla recommended tokens: {chinchilla_recommended_tokens:,}")
print(f"We are training with {total_tokens/chinchilla_recommended_tokens:.1%} of recommended tokens")

# Calculate effective training tokens over full training
batch_size = 16
max_steps = 20000
effective_tokens = batch_size * max_steps * (total_tokens/len(train_df))  # tokens seen during training

print(f"\nEffective training:")
print(f"Tokens seen during full training: {effective_tokens:,.0f}")
print(f"Each token seen average of {effective_tokens/total_tokens:.1f} times")

Total examples: 20808
Total tokens in training: 291312
Average tokens per example: 14.0

Chinchilla analysis:
Our parameters: 13,218,304
Our training tokens: 291,312
Chinchilla recommended tokens: 264,366,080
We are training with 0.1% of recommended tokens

Effective training:
Tokens seen during full training: 4,480,000
Each token seen average of 15.4 times
